# 验证并导入模型

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

# 验证PyTorch GPU可用性
print("PyTorch GPU可用：", torch.cuda.is_available())
print("CUDA版本：", torch.__version__)
# 验证Ultralytics版本及GPU适配
print("Ultralytics版本：", ultralytics.__version__)
# 加载YOLOv8n并指定GPU
model = YOLO("yolov8n.pt")
model.to("cuda:0")
print("YOLOv8n已加载到GPU：", next(model.model.parameters()).is_cuda)

# 开始训练

In [ ]:
from pathlib import Path
# 开始训练
# data: 从本项目目录启动 Jupyter，使用仓库内的相对路径配置
# epochs: 训练轮数
# imgsz: 输入图像大小
# batch: 批次大小
# device: 训练设备 (0 表示使用第一块 GPU)
results = model.train(
    data=str(Path("dataset.yaml").resolve()),
    epochs=120,
    imgsz=640,
    batch=64,
    device=0,
    project="butter_train_results",
    name="yolov8n_butter",
    exist_ok=True,  # 允许覆盖同名目录，方便调试
    verbose=True,  # 显示详细训练日志
)

In [ ]:
from ultralytics import YOLO

# 1. 加载训练好的最佳权重文件
# 注意：这里使用的是你刚才训练生成的 best.pt
model_path = "butter_train_results/yolov8n_butter/weights/best.pt"
model = YOLO(model_path)

# 2. 导出为 RKNN 兼容的 ONNX 模型
# 关键参数说明：
# format='rknn': 指定导出格式为 rknn (实际上会先导出为特殊的 onnx)
# opset=12: RKNN 通常推荐 opset 12 或 19，这里使用默认兼容性较好的配置
success = model.export(format="rknn")

print(f"导出完成: {success}")